# Практика: базовый RAG-pipeline (Jupyter Notebook)

Цель: собрать минимальный Retrieval-Augmented Generation (RAG) пайплайн для ответов на вопросы **по вашим документам**.

Что сделаем:
1) загрузим документы (txt/markdown или PDF),  
2) разобьём на чанки,  
3) построим эмбеддинги,  
4) проиндексируем во векторном хранилище (FAISS локально),  
5) реализуем retrieval + генерацию ответа LLM с цитированием источников.

> Этот ноутбук самодостаточен: можно запустить локально.  
> Для LLM/эмбеддингов есть два режима:
- **OpenAI** (нужен ключ в переменной окружения `OPENAI_API_KEY`)
- **Локальные эмбеддинги** через `sentence-transformers` (без ключа)

## 0) Установка зависимостей

Выполните одну из команд (в зависимости от ваших предпочтений).

**Вариант A (рекомендовано для старта):** LangChain + FAISS + sentence-transformers + PDF

In [1]:
# Если вы запускаете в чистом окружении:
# %pip install -U langchain langchain-community langchain-text-splitters faiss-cpu sentence-transformers pypdf

# Если планируете использовать OpenAI:
# %pip install -U langchain-openai openai

## 1) Импорт и конфигурация

Выберите провайдера эмбеддингов и LLM.

- `USE_OPENAI = True` - если есть ключ OpenAI
- `USE_OPENAI = False` - если хотите локальные эмбеддинги и (опционально) локальную LLM

In [10]:
import os
from typing import List, Dict, Any
from pathlib import Path



### 1.1) Провайдер эмбеддингов

In [ ]:
# import os # Import os module
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [2]:
from dotenv import load_dotenv
import os


load_dotenv('./.env')  # Загружаем переменные окружения из файла .env

True

In [3]:

# Выберите бэкенд LLM:
# - "openai"  -> ChatOpenAI (нужен OPENAI_API_KEY)
# - "ollama"  -> локальная модель через Ollama (нужен запущенный ollama serve)
# - "none"    -> без LLM (покажем только retrieved контекст)
LLM_BACKEND = os.getenv("LLM_BACKEND", "openai")  # openai | ollama | none

USE_OPENAI = (LLM_BACKEND == "openai")
USE_OLLAMA = (LLM_BACKEND == "ollama")

In [5]:
LLM_BACKEND, USE_OPENAI, USE_OLLAMA

('ollama', False, True)

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

if USE_OPENAI:
    from langchain_openai import OpenAIEmbeddings
    # Set the OPENAI_API_KEY as an environment variable before initializing OpenAIEmbeddings
    # The variable OPENAI_API_KEY is already loaded in the notebook's kernel state

    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
else:
    from langchain_community.embeddings import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/Users/stureiko/Documents/Programming/Proglib/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/lk/6f78t4jn60s5_ntqqc2dj0980000gn/T/ipykernel_93505/3309687347.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1321.21it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transf

### 1.2) Провайдер LLM

Если `USE_OPENAI=True`, используем OpenAI Chat model.
Если `USE_OPENAI=False`, используем заглушку, чтобы пайплайн retrieval был runnable и без LLM.

In [17]:
if USE_OPENAI:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"), temperature=0)

elif USE_OLLAMA:
    # pip install -U langchain-ollama
    # ollama pull llama3.1:8b-instruct
    # ollama serve   (если сервис ещё не запущен)
    from langchain_ollama import ChatOllama

    llm = ChatOllama(
        model=os.getenv("OLLAMA_MODEL", "llama3.2"),
        base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
        temperature=0,
    )

else:
    llm = None  # режим без LLM


## 2) Подготовка документов

Добавьте свои `.txt`/`.md`/`.pdf` файлы в папку `./data/`.

Если папки `data/` нет - создадим и положим демо-документ.

In [11]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

demo_path = DATA_DIR / "demo_policy.md"
if not demo_path.exists():
    demo_path.write_text(
        "# Политика отпусков (пример)\n"
        "Компания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.\n"
        "Сотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.\n"
        "Для оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.\n",
        encoding="utf-8"
    )

print("Файлы в ./data:")
for p in sorted(DATA_DIR.glob("*")):
    print(" -", p.name)

Файлы в ./data:
 - demo_policy.md


### 2.1) Загрузка документов

In [12]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_core.documents import Document # Corrected import path for Document

def load_documents(data_dir: Path) -> List[Document]:
    docs: List[Document] = []
    for path in data_dir.glob("*"):
        suf = path.suffix.lower()
        if suf in [".txt", ".md"]:
            docs.extend(TextLoader(str(path), encoding="utf-8").load())
        elif suf == ".pdf":
            docs.extend(PyPDFLoader(str(path)).load())
    return docs

docs = load_documents(DATA_DIR)
print(f"Загружено документов: {len(docs)}")
print("Пример:", docs[0].metadata)
print(docs[0].page_content[:400])

Загружено документов: 1
Пример: {'source': 'data/demo_policy.md'}
# Политика отпусков (пример)
Компания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.
Сотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.
Для оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.



## 3) Chunking (разбиение на фрагменты)

In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""],
)

chunks = text_splitter.split_documents(docs)
print(f"Чанков: {len(chunks)}")
print("Пример чанка:\n", chunks[0].page_content[:400])

Чанков: 1
Пример чанка:
 # Политика отпусков (пример)
Компания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.
Сотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.
Для оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.


## 4) Индексация во векторное хранилище (FAISS)

In [14]:
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# (опционально) сохранить индекс:
vectorstore.save_local("faiss_index")

print("Готово: индекс построен.")

Готово: индекс построен.


## 5) Retrieval: проверим, что поиск работает

In [15]:
def pretty_docs(docs: List[Any]) -> None:
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", None)
        where = f"{src}" + (f" (page {page})" if page is not None else "")
        print(f"\n--- [#{i}] {where} ---")
        print(d.page_content[:800])

question = "Сколько дней ежегодного оплачиваемого отпуска?"
hits = retriever.invoke(question) # Changed from get_relevant_documents

pretty_docs(hits)


--- [#1] data/demo_policy.md ---
# Политика отпусков (пример)
Компания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.
Сотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.
Для оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.


## 6) Генерация ответа без RAG

In [18]:
from langchain_core.messages import HumanMessage, SystemMessage

SYSTEM_PROMPT = (
    "Ты - ассистент, который отвечает на вопросы строго по предоставленному контексту.\n"
    "Правила:\n"
    "1) Не выдумывай факты.\n"
    "2) В конце добавь раздел 'Источники:' и перечисли источники (source/page).\n"
    "3) Если в источниках нет информации в разедел 'Источники:' добавь 'Базовая модель'"
)

def no_rag_answer(question: str) -> Dict[str, Any]:
    if llm is not None:
        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=f"Вопрос: {question}")
        ]
        resp = llm.invoke(messages)
        answer_text = resp.content
    else:
        answer_text = (
            "LLM не подключена (LLM_BACKEND=none)." )

    return {"question": question, "answer": answer_text}

result = no_rag_answer("Какая минимальная длина одной части отпуска?")
print(result["answer"])

Известно, что минимальная длина одной части отпуска не указана в законе. Однако, согласно статье 13 закона о правах потребителя (2019 года), минимальные размеры и сроки отпуска не указаны, но указано, что потребитель имеет право на получение информации о продукте или услуге.

Базовая модель: В законе нет конкретной информации о минимальной длине одной части отпуска.


## 7) Генерация ответа (RAG)

Соберём prompt так, чтобы модель отвечала строго по контексту и добавляла источники.

Если `USE_OPENAI=False`, вместо LLM вернём релевантный контекст (заглушка).

In [19]:
SYSTEM_PROMPT = (
    "Ты - ассистент, который отвечает на вопросы строго по предоставленному контексту.\n"
    "Правила:\n"
    "1) Если в контексте нет ответа - скажи: 'В документах нет ответа на этот вопрос.'\n"
    "2) Не выдумывай факты.\n"
    "3) В конце добавь раздел 'Источники:' и перечисли источники (source/page).\n"
)

def build_context(docs: List[Any]) -> str:
    parts = []
    for d in docs:
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", None)
        tag = f"{src}" + (f":{page}" if page is not None else "")
        parts.append(f"[{tag}]\n{d.page_content}")
    return "\n\n".join(parts)

def rag_answer(question: str) -> Dict[str, Any]:
    retrieved = retriever.invoke(question) # Changed from get_relevant_documents
    context = build_context(retrieved)

    if llm is not None:
        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=f"Вопрос: {question}\n\nКонтекст:\n{context}")
        ]
        resp = llm.invoke(messages)
        answer_text = resp.content
    else:
        answer_text = (
            "LLM не подключена (LLM_BACKEND=none). Ниже релевантный контекст, который можно подать в LLM:\n\n"
            + context[:1200]
            + ("\n\n... (контекст обрезан)" if len(context) > 1200 else "")
        )

    sources = [{"source": d.metadata.get("source", "unknown"), "page": d.metadata.get("page", None)} for d in retrieved]
    return {"question": question, "answer": answer_text, "sources": sources, "retrieved_docs": retrieved}

result = rag_answer("Какая минимальная длина одной части отпуска?")
print(result["answer"])

В документах нет ответа на этот вопрос.

Источники:
- [data/demo_policy.md]


## 7) Удобная функция для вопросов

In [20]:
def ask(q: str):
    res = rag_answer(q)
    print("\n=== Вопрос ===\n", res["question"])
    print("\n=== Ответ ===\n", res["answer"])

ask("За сколько дней до начала отпуска нужно подать заявление?")


=== Вопрос ===
 За сколько дней до начала отпуска нужно подать заявление?

=== Ответ ===
 В документах нет ответа на этот вопрос.

Источники:
- [data/demo_policy.md]


## 8) Агент для анализа договоров и уведомлений об изменениях (RAG + извлечение + сравнение)

Цель этого блока — показать **следующий шаг после базового RAG**: агент не просто отвечает на вопросы, а выполняет *действие* по документам:

1) берёт **контекстный документ** (политика / регламент) с изменением правила;  
2) находит в **трудовых договорах** релевантные условия (через RAG);  
3) извлекает **контакты и имена** (email, ФИО) и **значение условия**;  
4) сравнивает с новым правилом;  
5) формирует **черновик уведомления** (email) каждому сотруднику/ответственному.

> В демо мы **не отправляем письма** реально. Генерируем **draft**, который человек утверждает.


### 8.1) Пример документов (политика + договоры)

В реальном проекте вы загрузите PDF/DOCX/TXT.  
Для открытого урока удобно начать с текстовых примеров, чтобы сфокусироваться на логике агента.


In [ ]:
# Политика / регламент (контекстный документ)
policy_doc = """
Политика отпусков компании.
Версия: 2.1

Изменение:
Минимальная длительность ежегодного оплачиваемого отпуска должна быть НЕ МЕНЕЕ 10 календарных дней.
Данное требование применяется ко всем трудовым договорам.
""".strip()

# Примеры трудовых договоров (в реальности это будут файлы)
contracts = [
    {
        "doc_id": "contract_001",
        "text": """
ТРУДОВОЙ ДОГОВОР №001
Сотрудник: Иванов Иван Иванович
Email: ivanov.ii@company.ru

Раздел 5. Отпуск
5.1. Ежегодный оплачиваемый отпуск предоставляется продолжительностью не менее 7 календарных дней.
5.2. Остальные условия предоставления отпуска регулируются локальными нормативными актами.
""".strip()
    },
    {
        "doc_id": "contract_002",
        "text": """
ТРУДОВОЙ ДОГОВОР №002
Сотрудник: Петрова Мария Сергеевна
Контакт для уведомлений: maria.petrova@company.ru

Раздел 4. Отпуск
4.3. Ежегодный оплачиваемый отпуск предоставляется продолжительностью не менее 14 календарных дней.
""".strip()
    },
    {
        "doc_id": "contract_003",
        "text": """
ТРУДОВОЙ ДОГОВОР №003
Employee: John Smith
E-mail: john.smith@company.ru

Section: Vacation
The minimum annual paid vacation is 8 calendar days.
""".strip()
    },
]


### 8.2) Индексация: политика + договоры в общий векторный индекс

Подход для демо:
- один индекс,
- метаданные `doc_type` + `doc_id`,
- при retrieval мы дополнительно фильтруем результаты по `doc_id` (в FAISS нет встроенных фильтров).


In [ ]:
from langchain_core.documents import Document

all_docs = [Document(page_content=policy_doc, metadata={"doc_type": "policy", "doc_id": "policy_v2_1"})]
for c in contracts:
    all_docs.append(Document(page_content=c["text"], metadata={"doc_type": "contract", "doc_id": c["doc_id"]}))

# Переиспользуем text_splitter + embeddings из предыдущих шагов
chunks_all = text_splitter.split_documents(all_docs)

vectorstore_all = FAISS.from_documents(chunks_all, embeddings)

def retrieve_policy(query: str, k: int = 6):
    # Берём немного больше и фильтруем по doc_type
    docs = vectorstore_all.similarity_search(query, k=20)
    out = [d for d in docs if d.metadata.get("doc_type") == "policy"]
    return out[:k]

def retrieve_contract(doc_id: str, query: str, k: int = 6):
    docs = vectorstore_all.similarity_search(query, k=30)
    out = [d for d in docs if d.metadata.get("doc_type") == "contract" and d.metadata.get("doc_id") == doc_id]
    return out[:k]


### 8.3) Схемы структурированного извлечения (Pydantic)

Чтобы агент возвращал данные в предсказуемом виде:
- шаг 1: извлечь правило из политики;
- шаг 2: извлечь контакты и текущее значение условия из договора;
- шаг 3: сравнить (кодом);
- шаг 4: сформировать draft-письмо.


In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

class PolicyRule(BaseModel):
    clause: str = Field(description="Какое условие меняем (например, 'минимальная длительность отпуска')")
    min_days: int = Field(description="Новая минимальная длительность отпуска в днях")
    scope: str = Field(description="Кому применяется правило")

class ContractExtraction(BaseModel):
    doc_id: str
    employee_name: Optional[str] = Field(default=None, description="ФИО/имя сотрудника из договора (если найдено)")
    emails: List[str] = Field(default_factory=list, description="Все email адреса, явно присутствующие в договоре")
    vacation_min_days: Optional[int] = Field(default=None, description="Текущая минимальная длительность отпуска (если явно указана)")
    evidence: Optional[str] = Field(default=None, description="Короткий фрагмент текста договора, подтверждающий vacation_min_days")

class ChangeAssessment(BaseModel):
    doc_id: str
    employee_name: Optional[str]
    emails: List[str]
    current_min_days: Optional[int]
    required_min_days: int
    needs_change: bool
    reason: str


### 8.4) Извлекаем правило из политики (retrieval → structured output)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

policy_extractor = llm.with_structured_output(PolicyRule)

policy_prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты ассистент по HR/комплаенсу. Извлеки новое правило из политики. Возвращай только структурированные поля."),
    ("human", "Контекст (фрагменты политики):
{context}

Задание: извлеки новое правило про отпуск."),
])

policy_context_docs = retrieve_policy("минимальная длительность отпуска новое требование")
policy_context = "

---

".join([d.page_content for d in policy_context_docs])

rule = policy_extractor.invoke(policy_prompt.format_messages(context=policy_context))
rule


### 8.5) Извлечение из договора: контакты + текущее условие

Трюк для надёжности:
- retrieval ограничиваем *конкретным договором* (по `doc_id`),
- в системной инструкции явно запрещаем выдумывать значения.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

contract_extractor = llm.with_structured_output(ContractExtraction)

contract_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "Ты извлекаешь данные из трудового договора.
"
     "Если значение не найдено — верни null/None и НЕ выдумывай.
"
     "Emails извлекай только те, что явно присутствуют в тексте.
"
     "Если число дней указано словами, попытайся привести к числу, но только если уверен."
    ),
    ("human",
     "Контекст (фрагменты договора):
{context}

"
     "Задание:
"
     "- doc_id: {doc_id}
"
     "- извлеки имя/ФИО сотрудника
"
     "- emails (все email адреса)
"
     "- vacation_min_days: минимальная длительность отпуска (если явно указана)
"
     "- evidence: короткий фрагмент, подтверждающий vacation_min_days"
    ),
])

def extract_contract(doc_id: str) -> ContractExtraction:
    ctx_docs = retrieve_contract(doc_id, "отпуск минимальная длительность vacation minimum annual paid vacation", k=8)
    ctx = "

---

".join([d.page_content for d in ctx_docs])
    ce = contract_extractor.invoke(contract_prompt.format_messages(context=ctx, doc_id=doc_id))
    # На случай, если LLM вернул e-mail с лишними пробелами — чуть нормализуем
    ce.emails = [e.strip() for e in ce.emails if e and isinstance(e, str)]
    return ce

extractions = [extract_contract(c["doc_id"]) for c in contracts]
extractions


### 8.6) Сравнение правила с договором (детерминированная логика)

In [ ]:
def assess(rule: PolicyRule, ce: ContractExtraction) -> ChangeAssessment:
    req = int(rule.min_days)
    cur = ce.vacation_min_days

    if cur is None:
        # Не нашли явного условия — можно либо помечать как "требует ручной проверки",
        # либо считать, что изменений "не доказано".
        return ChangeAssessment(
            doc_id=ce.doc_id,
            employee_name=ce.employee_name,
            emails=ce.emails,
            current_min_days=None,
            required_min_days=req,
            needs_change=False,
            reason="В договоре не найдено явное условие о минимальной длительности отпуска (нужна ручная проверка)."
        )

    if cur < req:
        return ChangeAssessment(
            doc_id=ce.doc_id,
            employee_name=ce.employee_name,
            emails=ce.emails,
            current_min_days=cur,
            required_min_days=req,
            needs_change=True,
            reason=f"Текущее условие {cur} < требуемого {req}."
        )

    return ChangeAssessment(
        doc_id=ce.doc_id,
        employee_name=ce.employee_name,
        emails=ce.emails,
        current_min_days=cur,
        required_min_days=req,
        needs_change=False,
        reason=f"Текущее условие {cur} соответствует требованию (>= {req})."
    )

assessments = [assess(rule, ce) for ce in extractions]
assessments


### 8.7) Генерация черновиков уведомлений (draft)

Генерируем письма только для договоров, где `needs_change=True`.  
Если emails не извлечены — создаём draft без адресата (для ручного заполнения).


In [ ]:
email_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Ты помощник HR. Сформируй короткое и корректное уведомление.
"
     "Не давай юридических трактовок, не делай обещаний.
"
     "Тон: нейтральный, деловой.
"
     "Не упоминай, что ты ИИ."
    ),
    ("human",
     "Сформируй письмо сотруднику о необходимости обновить условие договора.

"
     "Данные:
"
     "- ФИО: {employee_name}
"
     "- Doc ID: {doc_id}
"
     "- Текущее условие (минимальный отпуск): {current_min_days}
"
     "- Новое требование (политика): минимум {required_min_days} дней
"
     "- Подтверждающий фрагмент (evidence): {evidence}

"
     "Сгенерируй:
"
     "1) Тему письма (одной строкой)
"
     "2) Тело письма (несколько абзацев)
"
     "Формат:
"
     "Subject: ...
"
     "Body:
"
     "...
"
    ),
])

def make_draft(a: ChangeAssessment, evidence: str | None):
    msgs = email_prompt.format_messages(
        employee_name=a.employee_name or "Коллега",
        doc_id=a.doc_id,
        current_min_days=a.current_min_days,
        required_min_days=a.required_min_days,
        evidence=evidence or "(не указан)",
    )
    txt = llm.invoke(msgs).content
    # очень простая разборка
    subject = "Уточнение условий трудового договора"
    body = txt
    m = re.search(r"Subject:\s*(.*)", txt)
    if m:
        subject = m.group(1).strip()
    m2 = re.search(r"Body:\s*(.*)", txt, flags=re.S)
    if m2:
        body = m2.group(1).strip()
    return {"to": a.emails, "subject": subject, "body": body}

email_drafts = {}
for a, ce in zip(assessments, extractions):
    if a.needs_change:
        email_drafts[a.doc_id] = make_draft(a, ce.evidence)

email_drafts


### 8.8) Заглушка "рассылки" (ничего реально не отправляем)

In [ ]:
def send_email_stub(to: list, subject: str, body: str):
    print("=== EMAIL DRAFT ===")
    print("To:", ", ".join(to) if to else "(no emails)")
    print("Subject:", subject)
    print(body)
    print()

for doc_id, draft in email_drafts.items():
    send_email_stub(draft.get("to", []), draft["subject"], draft["body"])


### 8.9) Чек-лист для «боевого» варианта (что сказать на открытом уроке)

**Надёжность и безопасность**
- PII (email/ФИО): минимизация, маскирование в логах, контроль доступа.
- Human-in-the-loop: письма только как draft, обязательное подтверждение человеком.
- Детерминированная бизнес-логика: сравнение условий делаем кодом, не LLM.

**Масштабирование**
- батчи и очереди (Celery/RQ/Kafka),
- хранение результатов (Postgres),
- аудит: кто/когда запустил анализ, что было отправлено/подтверждено.

**Качество**
- «золотой набор» договоров для тестов,
- метрики на извлечение (precision/recall) + регрессия промптов,
- обработка крайних случаев (несколько email, разные формулировки, отсутствие явного числа дней).
